In [1]:
import pandas as pd

cols = ["loan_amnt", "term", "int_rate", "emp_length", "home_ownership", "annual_inc",
        "purpose", "dti", "delinq_2yrs", "revol_util", "total_acc", "loan_status", "recoveries", "total_pymnt"]
df = pd.read_csv("data/raw/lending_club.csv", usecols=cols, low_memory=False)

df.info()
df["loan_status"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 14 columns):
 #   Column          Dtype  
---  ------          -----  
 0   loan_amnt       int64  
 1   term            str    
 2   int_rate        float64
 3   emp_length      str    
 4   home_ownership  str    
 5   annual_inc      float64
 6   loan_status     str    
 7   purpose         str    
 8   dti             float64
 9   delinq_2yrs     float64
 10  revol_util      float64
 11  total_acc       float64
 12  total_pymnt     float64
 13  recoveries      float64
dtypes: float64(8), int64(1), str(5)
memory usage: 241.5 MB


loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64

In [2]:
from pathlib import Path

# Keep loans with a resolved outcome only -- "Current," "Late," and "In Grace Period"
# loans haven't finished playing out yet and don't belong in a default-rate analysis.
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
df["bad_loan"] = (df["loan_status"] != "Fully Paid").astype(int)

df["term_months"] = df["term"].str.extract(r"(\d+)").astype(int)

# .astype(str) first because some Lending Club mirrors store emp_length/revol_util as
# text ("10+ years", "55.3%") and others as plain numbers -- casting to string before
# parsing means this line works either way instead of assuming one and crashing on the other.
df["emp_length"] = (df["emp_length"].astype(str).replace({"10+ years": "10", "< 1 year": "0"})
                     .str.extract(r"(\d+)", expand=False).astype(float))
df["revol_util"] = df["revol_util"].astype(str).str.rstrip("%").astype(float)

fill_cols = ["emp_length", "annual_inc", "revol_util", "total_acc", "delinq_2yrs"]
df[fill_cols] = df[fill_cols].fillna(df[fill_cols].median())

Path("data/processed").mkdir(parents=True, exist_ok=True)
df.to_csv("data/processed/lending_club_clean.csv", index=False)
df.shape

(1303638, 16)